# 🚗 CARLA AV Object Detection — YOLOv8 Training

**Dataset:** 28,814 images | **Classes:** 11

| ID | Class | Role |
|---|---|---|
| 0 | vehicle | Cars, trucks, buses |
| 1 | bike | Bicycles, motorbikes |
| 2 | traffic_light_red | 🔴 STOP |
| 3 | traffic_light_green | 🟢 GO |
| 4 | traffic_light_yellow | 🟡 SLOW DOWN |
| 5 | traffic_light_off | ⚫ Treat as STOP |
| 6 | speed_sign_30 | 🚦 30 km/h limit |
| 7 | speed_sign_60 | 🚦 60 km/h limit |
| 8 | speed_sign_90 | 🚦 90 km/h limit |
| 9 | traffic_sign | 🚧 General signs |
| 10 | pedestrian | 🚶 People |

---

##  Step 1 — Check GPU

In [ ]:
!nvidia-smi
import torch
print(f'\nCUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Sat Mar 14 18:24:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P8             34W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch
print(torch.cuda.get_device_name(0))
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

Tesla T4
VRAM: 15.6 GB


##  Step 2 — Install Dependencies

In [ ]:
!pip install ultralytics -q
import ultralytics
print(f'Ultralytics version: {ultralytics.__version__}')

Ultralytics version: 8.4.21


##  Step 3 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print(' Google Drive mounted!')

# Verify zip exists
import os
ZIP_PATH = '/content/drive/MyDrive/merged_carla_dataset.zip'
if os.path.exists(ZIP_PATH):
    size = os.path.getsize(ZIP_PATH) / 1e9
    print(f'Found merged_carla_dataset.zip ({size:.2f} GB)')
else:
    print('   merged_carla.zip not found!')
    print('   Make sure it is in the root of your Google Drive')
    print('   Or update ZIP_PATH if it is in a subfolder')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted!
✅ Found merged_carla_dataset.zip (1.18 GB)


## ✅ Step 4 — Unzip Dataset

In [ ]:
import os

ZIP_PATH    = '/content/drive/MyDrive/merged_carla_dataset.zip'
EXTRACT_DIR = '/content/merged_carla_dataset'

if not os.path.exists(EXTRACT_DIR):
    print(' Unzipping dataset (this may take a few minutes)...')
    !unzip -q "{ZIP_PATH}" -d /content/
    print('Unzip complete!')
else:
    print('Dataset already extracted — skipping unzip')

print('\n📁 Contents:')
!ls /content/merged_carla_dataset/

✅ Dataset already extracted — skipping unzip

📁 Contents:
data.yaml  images  labels


## ✅ Step 5 — Fix data.yaml for Colab

In [ ]:
import yaml

YAML_PATH = '/content/merged_carla_dataset/data.yaml'

yaml_content = {
    'path'  : '/content/merged_carla_dataset',
    'train' : 'images/train',
    'val'   : 'images/val',
    'test'  : 'images/test',
    'nc'    : 11,
    'names' : {
        0 : 'vehicle',
        1 : 'bike',
        2 : 'traffic_light_red',
        3 : 'traffic_light_green',
        4 : 'traffic_light_yellow',
        5 : 'traffic_light_off',
        6 : 'speed_sign_30',
        7 : 'speed_sign_60',
        8 : 'speed_sign_90',
        9 : 'traffic_sign',
        10: 'pedestrian',
    }
}

with open(YAML_PATH, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False, sort_keys=False)

print('data.yaml updated!')
with open(YAML_PATH) as f:
    print(f.read())

✅ data.yaml updated!
path: /content/merged_carla_dataset
train: images/train
val: images/val
test: images/test
nc: 11
names:
  0: vehicle
  1: bike
  2: traffic_light_red
  3: traffic_light_green
  4: traffic_light_yellow
  5: traffic_light_off
  6: speed_sign_30
  7: speed_sign_60
  8: speed_sign_90
  9: traffic_sign
  10: pedestrian



## ✅ Step 6 — Verify Dataset

In [ ]:
import os

base     = '/content/merged_carla_dataset'
expected = {'train': 23051, 'val': 2881, 'test': 2882}

print('Dataset Verification:')
print(f'{"Split":<8} {"Images":<10} {"Labels":<10} {"Expected":<10} {"Status"}')
print('-' * 55)

all_ok = True
for split in ['train', 'val', 'test']:
    imgs = len(os.listdir(f'{base}/images/{split}'))
    lbls = len(os.listdir(f'{base}/labels/{split}'))
    exp  = expected[split]
    ok   = 'Good' if imgs == exp else 'Bad'
    if imgs != exp:
        all_ok = False
    print(f'{split:<8} {imgs:<10} {lbls:<10} {exp:<10} {ok}')

print()
if all_ok:
    print(' All counts match! Ready to train.')
else:
    print('  Count mismatch — but training can still proceed.')

📊 Dataset Verification:
Split    Images     Labels     Expected   Status
-------------------------------------------------------
train    23051      23051      23051      ✅
val      2881       2881       2881       ✅
test     2882       2882       2882       ✅

✅ All counts match! Ready to train.


## ✅ Step 7 — Train YOLOv8
⏱️ Estimated time: **3–5 hours** on T4 GPU

In [ ]:
from ultralytics import YOLO
import shutil, os, glob

# ─── Drive backup setup ───────────────────────────────────────
DRIVE_BACKUP = '/content/drive/MyDrive/carla_av_weights'
os.makedirs(DRIVE_BACKUP, exist_ok=True)

def save_to_drive(trainer):
    """Auto-saves checkpoint to Drive after every epoch"""
    drive_backup = '/content/drive/MyDrive/carla_av_weights'
    epoch        = trainer.epoch + 1

    # Dynamically find the actual save directory
    weights_dir  = str(trainer.save_dir) + '/weights'

    print(f'Weights dir: {weights_dir}')  # debug line

    # Save every 5 epochs
    if epoch % 5 == 0:
        if os.path.exists(f'{weights_dir}/last.pt'):
            shutil.copy(f'{weights_dir}/last.pt', f'{drive_backup}/last.pt')
            shutil.copy(f'{weights_dir}/last.pt', f'{drive_backup}/epoch_{epoch}.pt')
            print(f' Epoch {epoch} — checkpoint saved to Drive!')
        else:
            print(f' last.pt not found at {weights_dir}')

    # Always save best.pt every epoch (not just every 5)
    if os.path.exists(f'{weights_dir}/best.pt'):
        shutil.copy(f'{weights_dir}/best.pt', f'{drive_backup}/best.pt')
        print(f' Epoch {epoch} — best.pt updated on Drive!')

# ─── Load model ──────────────────────────────────────────────
model = YOLO('yolov8m.pt')
model.add_callback('on_train_epoch_end', save_to_drive)

# ─── Train ───────────────────────────────────────────────────
results = model.train(
    data         = '/content/merged_carla_dataset/data.yaml',
    epochs       = 50,
    imgsz        = 512,
    batch        = 32,
    name         = 'carla_av_v1',
    exist_ok     = True,
    patience     = 15,
    augment      = True,
    optimizer    = 'AdamW',
    lr0          = 0.008,
    momentum     = 0.937,
    weight_decay = 0.0005,
    save         = True,
    save_period  = 5,
    plots        = True,
    device       = 0,
    workers      = 2,
    cache        = False,
    amp          = True,
)

print(' Training complete!')

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/merged_carla_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.008, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=carla_av_v12, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=1

## IF Colab Disconnect Start training continuing from last checkpoint

In [ ]:
import torch
import gc
import ctypes

# Step 1 — Kill all Python objects
gc.collect()
gc.collect()  # run twice

# Step 2 — Clear PyTorch cache
torch.cuda.empty_cache()
torch.cuda.synchronize()

# Step 3 — Reset all stats
torch.cuda.reset_peak_memory_stats()
torch.cuda.reset_accumulated_memory_stats()

# Step 4 — Delete any leftover model
try:
    del model
    gc.collect()
    torch.cuda.empty_cache()
except:
    pass

# Check result
free  = torch.cuda.mem_get_info()[0] / 1e9
total = torch.cuda.mem_get_info()[1] / 1e9
print(f'Free  : {free:.1f} GB')
print(f'Total : {total:.1f} GB')
print(f'Used  : {total-free:.1f} GB')

Free  : 15.5 GB
Total : 15.6 GB
Used  : 0.1 GB


In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

from ultralytics import YOLO
import shutil, os

DRIVE_BACKUP = '/content/drive/MyDrive/carla_av_weights'

def save_to_drive(trainer):
    drive_backup = '/content/drive/MyDrive/carla_av_weights'
    epoch        = trainer.epoch + 1
    weights_dir  = str(trainer.save_dir) + '/weights'

    # Save best.pt EVERY epoch
    if os.path.exists(f'{weights_dir}/best.pt'):
        shutil.copy(f'{weights_dir}/best.pt', f'{drive_backup}/best.pt')
        print(f'⭐ Epoch {epoch} — best.pt updated!')

    # Save last.pt + snapshot every 4 epochs
    if epoch % 4 == 0:
        if os.path.exists(f'{weights_dir}/last.pt'):
            shutil.copy(f'{weights_dir}/last.pt', f'{drive_backup}/last.pt')
            shutil.copy(f'{weights_dir}/last.pt', f'{drive_backup}/epoch_{epoch}.pt')
            print(f'💾 Epoch {epoch} — checkpoint saved to Drive!')

    # Print current learning rate every epoch
    current_lr = trainer.optimizer.param_groups[0]['lr']
    print(f' Epoch {epoch} — LR: {current_lr:.8f}')

model = YOLO(f'{DRIVE_BACKUP}/epoch_30.pt')
model.add_callback('on_train_epoch_end', save_to_drive)

model.train(
    data         = '/content/merged_carla_dataset/data.yaml',
    epochs       = 18,
    imgsz        = 512,
    batch        = 32,
    name         = 'carla_av_v1',
    exist_ok     = True,
    resume       = False,
    workers      = 2,
    cache        = False,
    amp          = True,
    save_period  = 4,         # ✅ every 4 epochs locally
    plots        = True,
    optimizer    = 'AdamW',
    lr0          = 0.001,
    lrf          = 0.01,
    momentum     = 0.937,
    weight_decay = 0.0005,
    warmup_epochs= 1,
    augment      = True,
    patience     = 15,
    hsv_h        = 0.015,
    hsv_s        = 0.7,
    hsv_v        = 0.4,
    fliplr       = 0.5,
    mosaic       = 1.0,
    close_mosaic = 5,
    mixup        = 0.1,
    copy_paste   = 0.1,
)

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=5, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/merged_carla_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=18, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=/content/drive/MyDrive/carla_av_weights/epoch_30.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=carla_av_v1, nbs=64, nms=False, opset=None, optimize=False, optim

In [ ]:
import os
path = '/content/drive/MyDrive/carla_av_weights'
files = os.listdir(path)
print(files)
# Should show: ['best.pt', 'last.pt']

## ✅ Step 8 — Evaluate on Test Set

In [ ]:
from ultralytics import YOLO

model   = YOLO('runs/detect/carla_av_v1/weights/best.pt')
metrics = model.val(
    data  = '/content/merged_carla_dataset/data.yaml',
    split = 'test'
)

print('\nTest Results:')
print(f'   mAP50      : {metrics.box.map50:.4f}')
print(f'   mAP50-95   : {metrics.box.map:.4f}')
print(f'   Precision  : {metrics.box.mp:.4f}')
print(f'   Recall     : {metrics.box.mr:.4f}')
print()
print('Per-class mAP50:')
classes = ['vehicle','bike','traffic_light_red','traffic_light_green',
           'traffic_light_yellow','traffic_light_off','speed_sign_30',
           'speed_sign_60','speed_sign_90','traffic_sign','pedestrian']
for i, (name, ap) in enumerate(zip(classes, metrics.box.ap50)):
    print(f'   [{i:2d}] {name:<25} {ap:.4f}')

## ✅ Step 9 — Save Weights to Google Drive

In [ ]:
import shutil, os

DRIVE_SAVE = '/content/drive/MyDrive/carla_av_weights'
os.makedirs(DRIVE_SAVE, exist_ok=True)

shutil.copy('runs/detect/carla_av_v1/weights/best.pt', f'{DRIVE_SAVE}/best.pt')
shutil.copy('runs/detect/carla_av_v1/weights/last.pt', f'{DRIVE_SAVE}/last.pt')

# Also save training plots
shutil.copytree(
    'runs/detect/carla_av_v1',
    f'{DRIVE_SAVE}/training_results',
    dirs_exist_ok=True
)

print(f' Saved to Google Drive → {DRIVE_SAVE}')
print('   best.pt           ← use for inference in CARLA')
print('   last.pt           ← use to resume training')
print('   training_results/ ← loss curves, confusion matrix')